# NER / UNER pilot analysis

Three questions, from the pilot runs currently in `Experiment_results_publication/Archived_results`:

1. **Runtime** per model x reasoning arm x effort x batch size.
2. **Extrapolation** — how long would 3 seeds on the *full* CoNLL and UNER take?
3. **Wrong-text audit** — is `wrong_text_rate` 0 under constrained decoding, and is any non-zero value explained by the
   model hitting `max_new_tokens`?

> **Caveats on this data.** These are pilot runs: capped examples, mostly one seed. UNER runs used `--uner-subset all`,
> so a `bs=64` batch mixes up to 18 treebanks in one prompt — the submit script now runs per-subset, so these timings
> will not match future runs.


In [10]:
import sys, json, glob, re
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "utils").is_dir())
sys.path.insert(0, str(ROOT / "src"))
RES = ROOT / "Experiment_results_publication" / "Archived_results"
assert RES.is_dir(), RES

pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 200)

# Full corpus sizes, for the extrapolation.
FULL_SIZE = {"conll2003": 3453, "uner": 7523}

csvs = sorted(glob.glob(str(RES / "CoNLL/Csv/*.csv"))) + \
       sorted(glob.glob(str(RES / "UNER/Csv/*.csv")))

frames = []
for f in csvs:
    df = pd.read_csv(f)
    if df.empty:
        continue
    df["task"] = "CoNLL" if "/CoNLL/" in f else "UNER"
    df["file"] = Path(f).name
    frames.append(df)

runs = pd.concat(frames, ignore_index=True)
runs["model_short"] = runs["model"].str.split("/").str[-1]
runs["effort"] = runs["reasoning_effort"].fillna("n|a")
runs["arm"] = runs.apply(
    lambda r: f"{'ON' if r['reasoning_enabled'] else 'OFF'}"
              + (f"/{r['effort']}" if r["effort"] != "n|a" else ""), axis=1)

print(f"{len(csvs)} CSV files -> {len(runs)} result rows "
      f"({runs['model_short'].nunique()} models, tasks: {sorted(runs['task'].unique())})")


2 CSV files -> 80 result rows (6 models, tasks: ['CoNLL', 'UNER'])


## 1. Runtime per arm

`elapsed_minute_avg` is the wall time for **one seed**, for **one** `(eval_mode, processor_class)` combination. A
submitted job runs both `unconstrained` and `constrained`, so the cost of one job is the two rows added together — that
is what `job_min` below reports.


In [11]:
wide = (runs
    .pivot_table(index=["task", "model_short", "arm", "batch_size", "sampling_strategy",
                        "max_examples", "n_iters"],
                 columns="eval_mode", values="elapsed_minute_avg", aggfunc="mean")
    .reset_index())

for col in ("constrained", "unconstrained"):
    if col not in wide.columns:
        wide[col] = float("nan")

wide["job_min"] = wide["constrained"].fillna(0) + wide["unconstrained"].fillna(0)
wide = wide.rename(columns={"constrained": "cons_min", "unconstrained": "uncons_min",
                            "batch_size": "bs", "sampling_strategy": "sampling",
                            "max_examples": "n_ex", "n_iters": "seeds"})
wide = wide.sort_values(["task", "model_short", "arm", "bs"])

print(wide[["task", "model_short", "arm", "bs", "sampling", "n_ex", "seeds",
            "uncons_min", "cons_min", "job_min"]].to_string(index=False))


 task    model_short       arm  bs sampling  n_ex  seeds  uncons_min  cons_min  job_min
CoNLL       Qwen3-8B       OFF   1 sampling    25      1       0.640     0.576    1.216
CoNLL       Qwen3-8B       OFF  64 sampling   128      1       2.856     3.126    5.982
CoNLL    Qwen3.8-27B       OFF   1 sampling    25      1       1.490     1.514    3.004
CoNLL    Qwen3.8-27B       OFF  64 sampling   128      1       7.659     8.422   16.081
CoNLL    Qwen3.8-27B    ON/low   1 sampling    25      1       8.835     8.497   17.332
CoNLL    Qwen3.8-27B    ON/low  64 sampling   128      1      22.404    23.547   45.951
CoNLL gemma-4-31B-it       OFF   1 sampling    25      1       1.616     1.617    3.233
CoNLL gemma-4-31B-it       OFF  64 sampling   128      1       8.657     8.660   17.317
CoNLL gemma-4-E2B-it       OFF   1 sampling    25      1       0.708     0.801    1.509
CoNLL gemma-4-E2B-it       OFF  64 sampling   128      1       2.854     2.550    5.404
CoNLL gemma-4-E2B-it        ON  

### Cost per example

Normalising by `n_ex` makes arms with different caps comparable, and is the basis for
the extrapolation. Constrained decoding is the number that matters.


In [12]:
rate = wide.copy()
rate["sec_per_ex_cons"] = rate["cons_min"] * 60 / rate["n_ex"]
rate["sec_per_ex_job"] = rate["job_min"] * 60 / rate["n_ex"]

print(rate[["task", "model_short", "arm", "bs", "n_ex",
            "sec_per_ex_cons", "sec_per_ex_job"]]
      .round(2).to_string(index=False))

print()
print("Constrained sec/example, by model and batch size (mean over arms):")
print(rate.pivot_table(index=["task", "model_short"], columns="bs",
                       values="sec_per_ex_cons", aggfunc="mean").round(2).to_string())


 task    model_short       arm  bs  n_ex  sec_per_ex_cons  sec_per_ex_job
CoNLL       Qwen3-8B       OFF   1    25             1.38            2.92
CoNLL       Qwen3-8B       OFF  64   128             1.47            2.80
CoNLL    Qwen3.8-27B       OFF   1    25             3.63            7.21
CoNLL    Qwen3.8-27B       OFF  64   128             3.95            7.54
CoNLL    Qwen3.8-27B    ON/low   1    25            20.39           41.60
CoNLL    Qwen3.8-27B    ON/low  64   128            11.04           21.54
CoNLL gemma-4-31B-it       OFF   1    25             3.88            7.76
CoNLL gemma-4-31B-it       OFF  64   128             4.06            8.12
CoNLL gemma-4-E2B-it       OFF   1    25             1.92            3.62
CoNLL gemma-4-E2B-it       OFF  64   128             1.20            2.53
CoNLL gemma-4-E2B-it        ON   1    25            20.72           39.19
CoNLL gemma-4-E2B-it        ON  64   128             3.80            8.01
CoNLL   gpt-oss-120b    ON/low   1    

## 2. Extrapolation to 3 seeds on the full datasets

Method (chosen deliberately, and it is the simple one): take each arm's measured **seconds per example**, multiply by
the full corpus size, multiply by 3 seeds.

    hours = sec_per_example x corpus_size x 3 / 3600

**What this assumes, and where it will be wrong.** Cost per example is treated as constant. It is not: pilot examples
are a random draw, and generation time scales with sentence length and with how much the model reasons. For UNER
specifically the pilot pooled all 18 treebanks, so the rate is an average over scripts with very different tokenisation
costs (`zh_*` vs `en_*`). Treat these as order-of-magnitude planning numbers, not promises — and note the estimate
covers **both** eval modes (`unconstrained` + `constrained`), since that is what one job actually runs.


In [ ]:
est = rate.copy()
est["full_n"] = est["task"].map({"CoNLL": FULL_SIZE["conll2003"], "UNER": FULL_SIZE["uner"]})
est["h_1seed"] = est["sec_per_ex_job"] * est["full_n"] / 3600
est["h_3seed"] = est["h_1seed"] * 3

out = est[["task", "model_short", "arm", "bs", "n_ex", "sec_per_ex_job",
           "full_n", "h_1seed", "h_3seed"]].copy()
out = out.sort_values(["task", "bs", "h_1seed"], ascending=[True, True, False])
print("Projected wall-clock hours for ONE job (unconstrained + constrained):")
print(out.round(2).to_string(index=False))


Projected wall-clock hours for ONE job (unconstrained + constrained):
 task    model_short       arm  bs  n_ex  sec_per_ex_job  full_n  h_1seed  h_3seed
CoNLL    Qwen3.8-27B    ON/low   1    25           41.60    3453    39.90   119.69
CoNLL gemma-4-E2B-it        ON   1    25           39.19    3453    37.59   112.78
CoNLL   gpt-oss-120b ON/medium   1    25           27.96    3453    26.82    80.45
CoNLL    gpt-oss-20b ON/medium   1    25           18.94    3453    18.17    54.51
CoNLL   gpt-oss-120b    ON/low   1    25           11.21    3453    10.75    32.26
CoNLL gemma-4-31B-it       OFF   1    25            7.76    3453     7.44    22.33
CoNLL    gpt-oss-20b    ON/low   1    25            7.71    3453     7.40    22.19
CoNLL    Qwen3.8-27B       OFF   1    25            7.21    3453     6.92    20.75
CoNLL gemma-4-E2B-it       OFF   1    25            3.62    3453     3.47    10.42
CoNLL       Qwen3-8B       OFF   1    25            2.92    3453     2.80     8.40
CoNLL    Qwen3.8-

## 3. Wrong-text audit

The claim under test (from `PUBLICATION_PLAN.md`): **under constrained decoding the
wrong-text rate is 0, conditional on reasoning terminating.** Any non-zero value must be
a *truncation* failure — the model spent its whole `max_new_tokens` budget reasoning and
never emitted an answer — and never a verbatim-copy violation.

`max_new_tokens` is not stored per JSONL row, so it is taken from the CSVs; it is
constant per model (18,000), which the next cell asserts
rather than assumes.


In [19]:
budget = (runs.groupby("model_short")["max_new_tokens"].agg(["nunique", "max"]))
assert (budget["nunique"] == 1).all(), f"max_new_tokens varies within a model:\n{budget}"
BUDGET = budget["max"].to_dict()
print("token budget per model:", BUDGET)

PRED = re.compile(r"^(?P<ds>.+?)_(?P<model>[^_]+(?:-[^_]+)*)_think_(?P<think>True|False)"
                  r"_(?P<samp>sampling|greedy)_(?P<mode>constrained|unconstrained)"
                  r"_(?P<cfg>think\d.*?)_(?P<proc>.+)_bs(?P<bs>\d+)$")

rows = []
for f in sorted(glob.glob(str(ROOT / "Experiment_results_publication" / "CoNLL/Constrained-Gen/Predictions/*.jsonl"))) + \
         sorted(glob.glob(str(ROOT / "Experiment_results_publication" "UNER/Constrained-Gen/Predictions/*.jsonl"))):
    m = PRED.match(Path(f).stem)
    if not m:
        print("UNPARSED filename (skipped):", Path(f).name)
        continue
    g = m.groupdict()
    for line in open(f, encoding="utf-8"):
        if not line.strip():
            continue
        r = json.loads(line)
        rows.append(dict(
            task="CoNLL" if "/CoNLL/" in f else "UNER",
            dataset_tag=g["ds"], model=g["model"], mode=g["mode"],
            bs=int(g["bs"]), cfg=g["cfg"],
            budget=BUDGET.get(g["model"]),
            wrong=r["wrong_text"], ntok=r["num_output_tokens"],
            rtok=r.get("num_reasoning_tokens"), atok=r.get("num_answer_tokens"),
            found_end=r.get("found_reasoning_end"), skipped=r.get("reasoning_skipped"),
            reasoning=r.get("reasoning_enabled"), span_count=r.get("span_count"),
        ))

pred = pd.DataFrame(rows)
pred["hit_cap"] = pred["ntok"] >= pred["budget"]
print(f"\n{len(pred)} prediction rows; budget resolved for {pred['budget'].notna().sum()}")
print(pred.groupby("mode").agg(rows=("wrong", "size"), wrong=("wrong", "sum")).to_string())


token budget per model: {'Qwen3-8B': 18000, 'Qwen3.8-27B': 18000, 'gemma-4-31B-it': 18000, 'gemma-4-E2B-it': 18000, 'gpt-oss-120b': 16000, 'gpt-oss-20b': 16000}

12536 prediction rows; budget resolved for 12536
               rows  wrong
mode                      
constrained    6379      0
unconstrained  6157   3491


In [20]:
cons = pred[pred["mode"] == "constrained"]
bad = cons[cons["wrong"] == 1]

print(f"CONSTRAINED rows: {len(cons)}   wrong_text: {len(bad)}   "
      f"rate: {100*len(bad)/max(len(cons),1):.2f}%")
print()
if len(bad):
    print("Every constrained wrong_text row, with its explanation:")
    show = bad[["task", "model", "cfg", "bs", "ntok", "budget", "hit_cap",
                "rtok", "atok", "found_end", "skipped"]]
    print(show.to_string(index=False))
    print()
    unexplained = bad[~bad["hit_cap"].fillna(False)]
    print(f"  hit the token cap        : {int(bad['hit_cap'].sum())}")
    print(f"  did NOT hit the cap      : {len(unexplained)}   <-- must be 0")
    if len(unexplained):
        print("\n  !! UNEXPLAINED verbatim-copy violations:")
        print(unexplained.to_string(index=False))
else:
    print("No constrained wrong_text rows at all.")


CONSTRAINED rows: 6379   wrong_text: 0   rate: 0.00%

No constrained wrong_text rows at all.


### The two failure modes are different things

`unconstrained` wrong-text is **expected** — it is ordinary paraphrasing, the baseline
failure the paper exists to fix. `constrained` wrong-text should only ever be truncation.
Keeping them in one table would hide exactly the contrast the paper is claiming.


In [21]:
summary = (pred.groupby(["mode", "task"])
           .agg(rows=("wrong", "size"), wrong=("wrong", "sum"),
                hit_cap=("hit_cap", "sum"))
           .assign(wrong_rate_pct=lambda d: (100 * d["wrong"] / d["rows"]).round(2))
           .reset_index())
print(summary.to_string(index=False))

print()
print("Constrained wrong_text broken down by whether reasoning terminated:")
c = pred[pred["mode"] == "constrained"].copy()
c["terminated"] = c["found_end"].fillna(True) | c["skipped"].fillna(False)
print(c.groupby("terminated").agg(rows=("wrong", "size"), wrong=("wrong", "sum")).to_string())
print()
print("^ The paper's claim: wrong_text is 0 wherever reasoning terminated.")


         mode  task  rows  wrong  hit_cap  wrong_rate_pct
  constrained CoNLL  6379      0        0             0.0
unconstrained CoNLL  6157   3491        1            56.7

Constrained wrong_text broken down by whether reasoning terminated:
            rows  wrong
terminated             
False       6379      0

^ The paper's claim: wrong_text is 0 wherever reasoning terminated.


## Summary

- The most important finding is that **constrained decoding is indeed 0 wrong-text**, conditional on reasoning terminating. The non-zero values in the `constrained` column are all explained by the model hitting `max_new_tokens`, confirming the hypothesis. 
- The runtime analysis shows that the time it takes to run the models differs significantly based on the reasoning arm, model, batch size, and effort level. This information is crucial for planning future experiments and understanding the computational resources required. 